<a href="https://colab.research.google.com/github/NaomiChoy03/undergrad-projects/blob/main/Naomi%20Choy/LoanApprovalXGBoostClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Get .csv dataset
url = 'https://raw.githubusercontent.com/NaomiChoy03/undergrad-projects/refs/heads/main/Naomi%20Choy/loan_approval_dataset.csv'
data = pd.read_csv(url, sep=r'\s*,\s*', header=0, encoding='ascii', engine='python')

data.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [40]:
# Cleaning data
data.drop('loan_id', axis=1, inplace=True)
# Filling in missing values
data.loc[data['residential_assets_value'] <= 0, 'residential_assets_value'] = data['residential_assets_value'].mean()
data.loc[data['commercial_assets_value'] <= 0, 'commercial_assets_value'] = data['commercial_assets_value'].mean()
data.loc[data['bank_asset_value'] <= 0, 'bank_asset_value'] = data['bank_asset_value'].mean()
# Encoding categorical variables
data = pd.get_dummies(data, columns=['education', 'self_employed', 'loan_status'], drop_first=True)
# Splitting the data into training and testing sets.


data.head()

<ipython-input-40-081d514f3323>:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7472616.537830873' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['residential_assets_value'] <= 0, 'residential_assets_value'] = data['residential_assets_value'].mean()
<ipython-input-40-081d514f3323>:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4973155.3056922' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['commercial_assets_value'] <= 0, 'commercial_assets_value'] = data['commercial_assets_value'].mean()
<ipython-input-40-081d514f3323>:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4976692.433825252' has dtype incompatible with int64, please 

,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,education_Not Graduate,self_employed_Yes,loan_status_Rejected
0,2,9600000,29900000,12,778,2400000.0,17600000.0,22700000,8000000.0,False,False,False
1,0,4100000,12200000,8,417,2700000.0,2200000.0,8800000,3300000.0,True,True,True
2,3,9100000,29700000,20,506,7100000.0,4500000.0,33300000,12800000.0,False,False,True
3,3,8200000,30700000,8,467,18200000.0,3300000.0,23300000,7900000.0,False,False,True
4,5,9800000,24200000,20,382,12400000.0,8200000.0,29400000,5000000.0,True,True,True


In [36]:
data.describe()

,no_of_dependents,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
count,4269.000000,4.269000e+03,4.269000e+03,4269.000000,4269.000000,4.269000e+03,4.269000e+03,4.269000e+03,4.269000e+03
mean,2.498712,5.059124e+06,1.513345e+07,10.900445,599.936051,7.601054e+06,5.097805e+06,1.512631e+07,4.986019e+06
std,1.695910,2.806840e+06,9.043363e+06,5.709187,172.430401,6.427739e+06,4.315951e+06,9.103754e+06,3.243022e+06
min,0.000000,2.000000e+05,3.000000e+05,2.000000,300.000000,1.000000e+05,1.000000e+05,3.000000e+05,1.000000e+05
25%,1.000000,2.700000e+06,7.700000e+06,6.000000,453.000000,2.400000e+06,1.500000e+06,7.500000e+06,2.400000e+06
50%,3.000000,5.100000e+06,1.450000e+07,10.000000,600.000000,5.900000e+06,4.000000e+06,1.460000e+07,4.600000e+06
75%,4.000000,7.500000e+06,2.150000e+07,16.000000,748.000000,1.130000e+07,7.600000e+06,2.170000e+07,7.100000e+06
max,5.000000,9.900000e+06,3.950000e+07,20.000000,900.000000,2.910000e+07,1.940000e+07,3.920000e+07,1.470000e+07


In [47]:
data.describe(exclude=np.number)

,education,self_employed,loan_status
count,4269,4269,4269
unique,2,2,2
top,Graduate,Yes,Approved
freq,2144,2150,2656
